# Darcy across a permeability jump

A vertical material interface separates permeabilities 1 and 1000. The exact pressure has a gradient jump, while the normal physical flux is continuous. The interface is aligned with macro and fine meshes; this is not an unfitted benchmark.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace, solve_darcy, solve_brinkman, solve_elasticity


In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "examples").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "examples"))
from plot_mesh import draw_macro_mesh

mesh=TriangleMesh.unit_square(2)
def permeability(x):
    """Return the scalar permeability in each material."""
    return np.where(x[:,0]<.5,1.,1000.)
def pressure(x):
    """Return a continuous piecewise affine pressure with unit horizontal flux."""
    return np.where(x[:,0]<=.5,1-x[:,0],.5-(x[:,0]-.5)/1000)
result=solve_darcy(mesh,permeability=permeability,dirichlet=pressure)
assert result.l2_error(pressure)<1e-11
assert result.flux_l2_error([1,0])<1e-9
print('pressure error',result.l2_error(pressure),'flux error',result.flux_l2_error([1,0]))
figure, axis = plt.subplots(figsize=(6, 5), constrained_layout=True)
for local,p in zip(result.local_meshes,result.pressure):
    artist = axis.tripcolor(local.points[:,0],local.points[:,1],local.cells,p,shading='gouraud',vmin=.4995,vmax=1)
figure.colorbar(artist, ax=axis, label='Pressure')
draw_macro_mesh(axis, mesh, label=True)
axis.legend(loc='lower center', bbox_to_anchor=(.5, 1.03))
axis.set(xlabel='x', ylabel='y', aspect='equal')
plt.show()

A material tensor must be symmetric positive definite at quadrature points. Resolving the true coefficient interfaces and oscillation length remains a discretization requirement.